
**Model:** DistilBERT (Pretrained) | **Metric:** MAP@3

Feed each (prompt, option) pair into DistilBERT → score → rank → top 3 = prediction.

In [1]:
!pip install transformers wandb --quiet

In [2]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
import wandb

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GENAI_T2_2026")

wandb.login(key = secret_value_0)

wandb.init(
    project = 'smart-mcq-solver',
    name = 'distilbert - v3',
    config  = {
        'model': 'distilbert-base-uncased',
        'max_len': 128,
        'batch_size' : 16,
        'epochs':10,
        'lr' : 2e-5,
    }
)
cfg = wandb.config

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260725_111208-cr1jmu2i
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run distilbert - v3
wandb: ⭐️ View project at https://wandb.ai/23f2003665-dl-genai-project/smart-mcq-solver
wandb: 🚀 View run at https://wandb.ai/23f2003665-dl-genai-project/smart-mcq-solver/runs/cr1jmu2i


In [4]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTIONS  = ['A', 'B', 'C', 'D', 'E']
print('Train:', train_df.shape, '| Test:', test_df.shape)

Train: (2000, 8) | Test: (500, 7)


In [5]:
# Split by question ID to avoid data leakage
all_ids = train_df['id'].unique()
np.random.shuffle(all_ids)
cut= int(len(all_ids) * 0.9)
train_data = train_df[train_df['id'].isin(all_ids[:cut])].reset_index(drop=True)
val_data= train_df[train_df['id'].isin(all_ids[cut:])].reset_index(drop=True)
print('Train:', len(train_data), '| Val:', len(val_data))

Train: 1800 | Val: 200


In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.df, self.tokenizer, self.max_len, self.is_test = df, tokenizer, max_len, is_test

    def __len__(self):
        return len(self.df) * 5

    def __getitem__(self, idx):
        row = self.df.iloc[idx // 5]
        opt = OPTIONS[idx % 5]
        enc = self.tokenizer(str(row['prompt']), str(row[opt]),
                                max_length=self.max_len, padding='max_length',
                                truncation=True, return_tensors='pt')
        
        item   = {'input_ids' : enc['input_ids'].squeeze(),
                  'attention_mask': enc['attention_mask'].squeeze()}
        
        if not self.is_test:
            item['label'] = torch.tensor(1.0 if row['answer'] == opt else 0.0)
        return item

In [7]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model)
model = AutoModelForSequenceClassification.from_pretrained(cfg.model, num_labels=1).to(device)
optimizer = AdamW(model.parameters(), lr=cfg.lr)
loss_fn = torch.nn.BCEWithLogitsLoss()

train_loader = DataLoader(MCQDataset(train_data, tokenizer, cfg.max_len), batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(MCQDataset(val_data,   tokenizer, cfg.max_len), batch_size=cfg.batch_size)
print('Model loaded. Train batches:', len(train_loader))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded. Train batches: 563


In [8]:
def compute_map3(df, flat_scores):
    scores = flat_scores.reshape(-1, 5)
    aps = []
    for i in range(len(df)):
        correct = OPTIONS.index(df.iloc[i]['answer'])
        top3 = np.argsort(-scores[i])[:3]
        ap, hits = 0.0, 0
        for rank, idx in enumerate(top3):
            if idx == correct:
                hits += 1
                ap += hits / (rank + 1)
        aps.append(ap)
    return float(np.mean(aps))

In [ ]:
for epoch in range(cfg.epochs):
    # Training model
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['label'].to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(ids, mask).logits.squeeze(-1), lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()


    # Validating 
    model.eval()
    scores = []
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            scores.extend(torch.sigmoid(model(ids, mask).logits.squeeze(-1)).cpu().numpy())

    val_map3 = compute_map3(val_data, np.array(scores))
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val MAP@3: {val_map3:.4f}')
    wandb.log({'epoch': epoch+1, 'train_loss': avg_loss, 'val_map3': val_map3})

torch.save(model.state_dict(), '/kaggle/working/distilbert.pt')
print('Model saved.')

Epoch 1 | Loss: 0.4558 | Val MAP@3: 0.9225
Epoch 2 | Loss: 0.2244 | Val MAP@3: 0.9783
Epoch 3 | Loss: 0.1118 | Val MAP@3: 0.9875
Epoch 4 | Loss: 0.0768 | Val MAP@3: 0.9975
Epoch 5 | Loss: 0.0700 | Val MAP@3: 0.9950
Epoch 6 | Loss: 0.0573 | Val MAP@3: 1.0000
Epoch 7 | Loss: 0.0349 | Val MAP@3: 0.9975
Epoch 8 | Loss: 0.0317 | Val MAP@3: 0.9975
Epoch 9 | Loss: 0.0248 | Val MAP@3: 0.9950
Epoch 10 | Loss: 0.0136 | Val MAP@3: 1.0000
Model saved.


In [ ]:
# Inference 
model.eval()
test_loader = DataLoader(MCQDataset(test_df, tokenizer, cfg.max_len, is_test=True), batch_size=cfg.batch_size)
test_scores = []
with torch.no_grad():
    for batch in test_loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        test_scores.extend(torch.sigmoid(model(ids, mask).logits.squeeze(-1)).cpu().numpy())

test_scores = np.array(test_scores).reshape(-1, 5)

# Submission 
rows = []
for i in range(len(test_df)):
    top3 = [OPTIONS[j] for j in np.argsort(-test_scores[i])[:3]]
    rows.append({'id': test_df.iloc[i]['id'], 'Prediction': ' '.join(top3)})

pd.DataFrame(rows).to_csv('/kaggle/working/submission_distilbert.csv', index=False)
print('Done. Sample:')
pd.DataFrame(rows).head()

Done. Sample:


,id,Prediction
0,1,A C D
1,2,B E C
2,3,B A C
3,4,E C B
4,5,C A E


In [11]:
wandb.finish()

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:      epoch ▁▂▃▃▄▅▆▆▇█
wandb: train_loss █▄▃▂▂▂▁▁▁▁
wandb:   val_map3 ▁▆▇███████
wandb: 
wandb: Run summary:
wandb:      epoch 10
wandb: train_loss 0.01364
wandb:   val_map3 1
wandb: 
wandb: 🚀 View run distilbert - v3 at: https://wandb.ai/23f2003665-dl-genai-project/smart-mcq-solver/runs/cr1jmu2i
wandb: ⭐️ View project at: https://wandb.ai/23f2003665-dl-genai-project/smart-mcq-solver
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260725_111208-cr1jmu2i/logs
